In [ ]:
import os
import sys
import numpy as np

if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from utils import build_BBP_cell

from neuron import h
h.load_file('stdrun.hoc')
h.celsius = 34

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

fontsize = 9
lw = 0.75
matplotlib.rc('font', **{'family': 'Arial', 'size': fontsize})
matplotlib.rc('axes', **{'linewidth': 0.75, 'labelsize': fontsize})
matplotlib.rc('xtick', **{'labelsize': fontsize})
matplotlib.rc('ytick', **{'labelsize': fontsize})
matplotlib.rc('xtick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.major', **{'width': lw, 'size':3})
matplotlib.rc('ytick.minor', **{'width': lw, 'size':1.5})

In [ ]:
layer = 5
if layer == 23:
    model_name = 'L23_PC_cADpyr229_1'
elif layer == 5:
    model_name = 'L5_TTPC1_cADpyr232_1'
else:
    raise Exception('Only layers 2/3 and 5 are currently supported')
model_dir = os.path.join('..', 'BBP_models', model_name)
cell = build_BBP_cell(model_dir, add_synapses=0, replace_axon=1)

In [ ]:
rec = {'time': h.Vector(), 'soma(0.5)': h.Vector()}
rec['time'].record(h._ref_t)
rec['soma(0.5)'].record(cell.soma[0](0.5)._ref_v)

In [ ]:
delay = 3000
stim_dur = 500
after = 200
stim = h.IClamp(cell.soma[0](0.5))
stim.delay = delay
stim.dur = stim_dur
stim.amp = 0.5

In [ ]:
h.tstop = delay + stim_dur + after
h.cvode_active(1)
h.run()

In [ ]:
t,Vm = np.array(rec['time']), np.array(rec['soma(0.5)'])
idx = (t > delay-100) & (t < delay+stim_dur+200)
fig,ax = plt.subplots(1, 1, figsize=(5,3.5))
ax.plot(t[idx]-delay, Vm[idx], 'k', lw=1)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Vm (mV)')
ax.grid(which='major', axis='y', lw=0.5, ls=':', color=[.6,.6,.6])
sns.despine()
fig.tight_layout()